# Chatbots — Rule-Based to Neural to LLM Agents Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: rule-based pattern matching

In [ ]:
```python

import re

class RulePattern:

    def __init__(self, pattern, response_template):

        self.regex = re.compile(pattern, re.IGNORECASE)

        self.template = response_template

PATTERNS = [

    RulePattern(r"my name is (\w+)", "Nice to meet you, {0}."),

    RulePattern(r"i (need|want) (.+)", "Why do you {0} {1}?"),

    RulePattern(r"i feel (.+)", "Why do you feel {0}?"),

    RulePattern(r"(.*)", "Tell me more about that."),

]

def rule_based_respond(user_input):

    for pattern in PATTERNS:

        m = pattern.regex.match(user_input.strip())

        if m:

            return pattern.template.format(*m.groups())

    return "I don't understand."

In [ ]:
```

ELIZA in 20 lines. The reflection trick ("I feel sad" → "Why do you feel sad") is the canonical psychotherapist demo from Weizenbaum 1966. Still instructive.

### Step 2: retrieval-based (FAQ)

This illustrative snippet requires `pip install sentence-transformers` (which pulls in torch). The runnable `code/main.py` for this lesson uses a stdlib Jaccard similarity instead, so the lesson runs without external dependencies.

In [ ]:
```python

from sentence_transformers import SentenceTransformer

import numpy as np

FAQ = [

    ("how do i reset my password", "Go to Settings > Security > Reset Password."),

    ("how do i cancel my order", "Go to Orders, find the order, click Cancel."),

    ("what is your return policy", "30-day returns on unused items, original packaging."),

]

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

faq_questions = [q for q, _ in FAQ]

faq_embeddings = encoder.encode(faq_questions, normalize_embeddings=True)

def faq_respond(user_input, threshold=0.5):

    q_emb = encoder.encode([user_input], normalize_embeddings=True)[0]

    sims = faq_embeddings @ q_emb

    best = int(np.argmax(sims))

    if sims[best] < threshold:

        return None

    return FAQ[best][1]

In [ ]:
```

Threshold-based refusal is the key design choice. If the best match is not close enough, return `None` and let the system escalate.

### Step 3: neural generation (baseline)

Use a small instruction-tuned encoder-decoder (FLAN-T5) or a fine-tuned conversational model. Production-unusable on its own in 2026 (contradiction, off-topic drift, factual nonsense), but ships inside hybrid systems for natural phrasing. DialoGPT-style decoder-only models need explicit turn separators and EOS handling to produce coherent replies; a FLAN-T5 text2text pipeline works out of the box for a teaching example.

In [ ]:
```python

from transformers import pipeline

chatbot = pipeline("text2text-generation", model="google/flan-t5-small")

response = chatbot("Respond politely to: Hi there!", max_new_tokens=40)

print(response[0]["generated_text"])

In [ ]:
```

### Step 4: LLM agent loop

The 2026 production shape:

In [ ]:
```python

def agent_loop(user_message, tools, llm, max_steps=5):

    history = [{"role": "user", "content": user_message}]

    for _ in range(max_steps):

        response = llm(history, tools=tools)

        tool_call = response.get("tool_call")

        if tool_call:

            tool_name = tool_call.get("name")

            args = tool_call.get("arguments")

            if not isinstance(tool_name, str) or tool_name not in tools:

                history.append({"role": "assistant", "tool_call": tool_call})

                history.append({"role": "tool", "name": str(tool_name), "content": f"error: unknown tool {tool_name!r}"})

                continue

            if not isinstance(args, dict):

                history.append({"role": "assistant", "tool_call": tool_call})

                history.append({"role": "tool", "name": tool_name, "content": f"error: arguments must be a dict, got {type(args).__name__}"})

                continue

            fn = tools[tool_name]

            result = fn(**args)

            history.append({"role": "assistant", "tool_call": tool_call})

            history.append({"role": "tool", "name": tool_name, "content": result})

        else:

            return response["content"]

    return "I could not complete the task in the step budget."

In [ ]:
```

Three things to name. Tools are callable functions the LLM can invoke. The loop terminates when the LLM returns a final answer instead of a tool call. The step budget prevents infinite loops on ambiguous tasks.

Real production adds: retrieval-first grounding (inject relevant docs before each LLM call), guardrails (refuse destructive actions without confirmation), observability (log every step), and evaluations (automated checks that agent behavior stays on-spec).

### Step 5: hybrid routing

In [ ]:
```python

def hybrid_chat(user_input):

    if is_destructive_action(user_input):

        return structured_flow(user_input)

    faq_answer = faq_respond(user_input, threshold=0.6)

    if faq_answer:

        return faq_answer

    return agent_loop(user_input, tools, llm)

def is_destructive_action(text):

    danger_words = ["delete", "cancel", "charge", "refund", "transfer"]

    return any(w in text.lower() for w in danger_words)

In [ ]:
```

The pattern: deterministic rules for anything destructive, retrieval for canned FAQs, LLM agents for everything else. This is what ships in 2026 customer-support systems.

## Exercises

In [ ]:
1. **Easy.** Implement the rule-based respond above with 10 patterns for a coffee-shop ordering bot. Test edge cases: double orders, modifications, cancellation, unclear intent.
2. **Medium.** Build a hybrid FAQ + LLM fallback. 50 canned FAQ entries for a SaaS product, LLM fallback with retrieval over the docs site. Measure refusal rate and accuracy on 100 real support questions.
3. **Hard.** Implement the agent loop above with three tools (search, read-user-data, send-email). Run an evaluation with 50 test scenarios including prompt injection attempts. Report off-task rate, failed task rate, and any injection success.